# Explanation Generation
I need to use the model to get *k* explanations for each line of generated code. Firtly, I need to delete the comments line from my dataset.

In [1]:
import re
import os
import torch
import pandas as pd
from tqdm import tqdm
from enum import Enum
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
python_csv_file = '/datasets/selected_problems_python_14B_no_comments_final.csv'
c_sharp_csv_file = '/datasets/selected_problems_c_sharp_14B_no_comments_final.csv'
typescript_csv_file = '/datasets/selected_problems_typescript_14B_no_comments_final.csv'

def get_generated_code(csv_file):
    csv_full_path = os.getcwd()+csv_file
    if os.path.exists(csv_full_path):
        return pd.read_csv(csv_full_path)
    else:
        print("File not found.")
        return None
    
python_solutions = get_generated_code(python_csv_file)
c_sharp_solutions = get_generated_code(c_sharp_csv_file)
typescript_solutions = get_generated_code(typescript_csv_file)

In [3]:
class Language(Enum):
    PYTHON = "python"
    TYPESCRIPT = "typescript"
    C_SHARP = "csharp"

In [4]:
model_name = "Qwen/Qwen2.5-Coder-14B-Instruct"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)#.to(device)

cuda


2025-05-03 09:59:57.058932: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-03 09:59:57.106574: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-03 09:59:57.106611: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-03 09:59:57.106655: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-03 09:59:57.118001: I tensorflow/core/platform/cpu_feature_g

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [17]:
def add_line_numbers(code):
    lines = code.split('\n')
    numbered_lines = [f"{i + 1}. {line}" for i, line in enumerate(lines)]
    return '\n'.join(numbered_lines), len(lines)
print(add_line_numbers(python_solutions[0:1]["solution_no_comments"].to_list()[0])[0])

1. def read_input():
2.     import sys
3.     input = sys.stdin.read
4.     data = input().split()
5.     index = 0
6.     d = int(data[index])
7.     index += 1
8.     datasets = []
9.     for _ in range(d):
10.         n = int(data[index])
11.         index += 1
12.         scores = list(map(int, data[index:index + n]))
13.         index += n
14.         datasets.append(scores)
15.     return datasets
16. def competition_ranking(scores):
17.     sorted_scores = sorted([(score, i) for i, score in enumerate(scores)], reverse=True)
18.     ranks = [0] * len(scores)
19.     current_rank = 1
20.     for i, (_, original_index) in enumerate(sorted_scores):
21.         if i > 0 and sorted_scores[i][0] != sorted_scores[i - 1][0]:
22.             current_rank = i + 1
23.         ranks[original_index] = current_rank
24.     return ranks
25. def dense_ranking(scores):
26.     sorted_scores = sorted([(score, i) for i, score in enumerate(scores)], reverse=True)
27.     ranks = [0] * len(scores)
28

In [41]:
max_context_length = model.config.max_position_embeddings

def get_generated_explanations(programming_language, problem_description, code_solution):
    code_solution_with_lines, lines_number = add_line_numbers(code_solution.strip())
    # print(code_solution_with_lines)
    explanation_prompt = f"""Read the given problem and the proposed {programming_language.value} solution. For **each line** of the code, provide a detailed explanation. Please ensure the following:

        - **Do not add any extra text** such as test cases, summaries, or comments.
        - Do not merge explanations for multiple lines.
        - Even if two lines seem similar, explain them individually.
        - Do not exclude any line. TIPS: You have {lines_number} lines in total.

        Output format: 
        [Line x]: Explanation: <your explanation>
        
        Problem: {problem_description}
        Code: ```{code_solution_with_lines}```
        """
    # print(explanation_prompt)
    explanation_inputs = tokenizer(explanation_prompt, return_tensors="pt", padding=True)
    explanation_outputs = model.generate(input_ids=explanation_inputs['input_ids'], attention_mask=explanation_inputs['attention_mask'], max_length=max_context_length+1)
    return tokenizer.decode(explanation_outputs[0], skip_special_tokens=True)[len(explanation_prompt)-1:].strip()

In [31]:
columns = ['title', 'explanation']
python_explanations = pd.DataFrame([], columns=columns)
c_sharp_explanations = pd.DataFrame([], columns=columns)
typescript_explanations = pd.DataFrame([], columns=columns)

In [32]:
def add_explanations(solutions, programming_language, explanations):
    for index, row in tqdm(solutions.iterrows()):
         explanations.loc[len(explanations)] = [row['title'], get_generated_explanations(programming_language, row['description'], row['solution_no_comments'])]

In [36]:
python_solutions[0:1]

,title,description,difficulty,source,solution,solution_no_comments
0,Sports Rank,Sports leagues rank their participants (teams ...,Medium,Kattis,def read_input():\n import sys\n input =...,def read_input():\n import sys\n input =...


In [42]:
add_explanations(python_solutions[0:1], Language.PYTHON, python_explanations)

1it [19:59, 1199.94s/it]


In [39]:
python_explanations

,title,explanation
0,Sports Rank,[Line 1]: Explanation: Define a function named...


In [40]:
print(python_explanations[0:1]["explanation"].to_list()[0])

[Line 1]: Explanation: Define a function named `read_input` to read input data.
         [Line 2]: Explanation: Import the `sys` module to access system-specific parameters and functions.
         [Line 3]: Explanation: Assign the `read` method of `sys.stdin` to the variable `input` to read all input at once.
         [Line 4]: Explanation: Split the input string into a list of strings `data` using whitespace as the delimiter.
         [Line 5]: Explanation: Initialize an index variable `index` to 0 to keep track of the current position in the `data` list.
         [Line 6]: Explanation: Convert the first element of `data` (at index 0) to an integer and assign it to `d`, representing the number of data sets.
         [Line 7]: Explanation: Increment the `index` by 1 to move to the next element in `data`.
         [Line 8]: Explanation: Initialize an empty list `datasets` to store the scores for each data set.
         [Line 9]: Explanation: Start a loop that will iterate `d` times to p

**Now, I need to run these method for each solution file I generated.**

In [89]:
columns = ['title', 'explanation']
python_explanations = pd.DataFrame([], columns=columns)
c_sharp_explanations = pd.DataFrame([], columns=columns)
typescript_explanations = pd.DataFrame([], columns=columns)

In [90]:
add_explanations(python_solutions, Language.PYTHON, python_explanations)

30it [6:10:57, 741.93s/it] 


In [91]:
add_explanations(c_sharp_solutions, Language.C_SHARP, c_sharp_explanations)

30it [6:59:36, 839.20s/it] 


In [92]:
add_explanations(typescript_solutions, Language.TYPESCRIPT, typescript_explanations)

30it [6:32:44, 785.47s/it]


In [83]:
python_explanations

,title,explanation
0,Relative Ranks,[Line 1]: Explanation: Define a function named...
1,Truncate Sentence,```\n: [Line 1]: Explanation: Define a functio...
2,Perform String Shifts,[Line 1]: Explanation: Define a function named...
3,Invert Binary Tree,``` [Line 1]: Explanation: Defines the TreeNod...
4,Merge Strings Alternately,[Line 1]: Explanation: Define a function named...


# Line Explanation Number Checking
To make sure that each line was explained, I need to check the number of code lines with the number of explanation lines.

In [84]:
def check_explanations_for_lines(code, explanations):
    code_lines = [line for line in code.split('\n') if line.strip()]
    explanation_lines = [line for line in explanations.split('\n') if line.strip().startswith("[Line")]
    
    return len(code_lines) == len(explanation_lines)

In [75]:
check_explanations_for_lines(python_solutions[0:1]["solution_no_comments"].to_list()[0], python_explanations[0:1]["explanation"].to_list()[0])

True

In [85]:
def check_all_explanations(solutions, explanations):
    for index, row in tqdm(solutions.iterrows()):
        if check_explanations_for_lines(row["solution_no_comments"], explanations.loc[index, "explanation"]):
            continue
        else:
            print(row["title"])

In [86]:
check_all_explanations(python_solutions[0:5], python_explanations)

5it [00:00, 2803.68it/s]

Truncate Sentence
Invert Binary Tree


In [93]:
python_explanations.to_csv('datasets/python_14B_explanations_final_1.csv', index=False)

In [94]:
c_sharp_explanations.to_csv('datasets/c_sharp_14B_explanations_final_1.csv', index=False)

In [95]:
typescript_explanations.to_csv('datasets/typescript_14B_explanations_final_1.csv', index=False)